## RAG as a graph

- A chain retrieves once and answers ; a graph can judge its answer and retry
- The evaluation node is what makes this a graph rather than a chain
- The `attempts` guard is what stops it retrying for ever

Retrieval, generation, evaluation, and a conditional edge back to retrieval.

In [ ]:
!pip install --quiet langchain-community faiss-cpu

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

### Preparing the document database

In [3]:
from typing import TypedDict

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, StateGraph

docs = [
    "LangChain is a framework for working with large language models.",
    "LangGraph is a framework that builds workflows as state graphs.",
    "Retrieval-Augmented Generation combines context retrieval with answer generation.",
    "FAISS is a library for finding the nearest vectors among embeddings.",
]

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
splits = splitter.create_documents(docs)

vectorstore = FAISS.from_documents(splits, embedding=make_embeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"{len(splits)} chunks indexed")

C:\Users\feiko\AppData\Local\Temp\ipykernel_36960\719414661.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


4 chunks indexed


### State

In [4]:
class State(TypedDict):
    question: str
    context: str
    answer: str
    pass_eval: bool
    attempts: int

### Retrieval node

In [5]:
def retrieval(state: State) -> State:
    hits = retriever.invoke(state["question"])
    return {"context": "\n".join(d.page_content for d in hits)}

### Generation node

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer from the CONTEXT only. If it is not there, say so.\n\nCONTEXT:\n{context}"),
    ("user", "{question}"),
])


def generation(state: State) -> State:
    response = (prompt | llm).invoke({
        "context": state["context"],
        "question": state["question"],
    })
    return {"answer": response.content, "attempts": state.get("attempts", 0) + 1}

### Evaluation node

The node that makes this a graph rather than a chain: it decides whether the
answer goes forward or goes back.

In [7]:
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", "You judge answers. Reply with one word, yes or no."),
    ("user", "Question: {question}\nAnswer: {answer}\n"
             "Does the answer actually answer the question?"),
])


def eval_node(state: State) -> State:
    response = (eval_prompt | llm).invoke({
        "question": state["question"],
        "answer": state["answer"],
    })
    return {"pass_eval": response.content.strip().lower().startswith("yes")}

### Final node

In [8]:
def finish(state: State) -> State:
    if state["pass_eval"]:
        print(f"approved after {state['attempts']} attempt(s): {state['answer']}")
    else:
        print(f"gave up after {state['attempts']} attempt(s)")
    return state

### Graph structure

`check_eval` is the conditional edge. Note the `attempts` guard: without it a
model that never satisfies its own judge loops until the recursion limit stops
the graph.

In [9]:
graph = StateGraph(State)

graph.add_node("retrieval", retrieval)
graph.add_node("generation", generation)
graph.add_node("eval", eval_node)
graph.add_node("finish", finish)

graph.set_entry_point("retrieval")
graph.add_edge("retrieval", "generation")
graph.add_edge("generation", "eval")


def check_eval(state: State) -> str:
    if state["pass_eval"] or state["attempts"] >= 3:
        return "finish"
    return "generation"


graph.add_conditional_edges("eval", check_eval, ["finish", "generation"])
graph.add_edge("finish", END)

app = graph.compile()

### Run graph

In [ ]:
result = app.invoke({"question": "When must a supplier invoice be paid?", "attempts": 0})
print()
print("pass_eval:", result["pass_eval"])

## Tasks

1. Replace the texts in *Preparing the document database* with our own, and
   change the question in *Run graph* to match them.

2. Ask a question the documents cannot answer. Watch `attempts` climb and the
   guard stop the loop. What would happen without the guard?

3. Remove `eval_node` from *Graph structure* and wire `generation` straight
   to `finish`. Compare the answers with and without the evaluation node, and
   decide whether the extra model call earned its keep.